In [1]:
import numpy as np
from kinisi.parser import _get_matrix

In [2]:
def get_disp(coords, latt):
   """
   Calculate displacements for NVT trajectories.

   :param coords: Fractional coordinates for all atoms.
   :param latt: Lattice descriptions.

   :return: Numpy array of with shape [site, time step, axis] describing displacements.
   """
   coords = np.concatenate(coords, axis=1)
   d_coords = coords[:, 1:] - coords[:, :-1]
   d_coords = d_coords - np.round(d_coords)
   f_disp = np.cumsum(d_coords, axis=1)
   latt = np.array(latt)
   disp = np.einsum('ijk,jkl->ijk', f_disp, latt[1:])
   return disp


def get_disp_npt(coords, latt):
   """
   Calculate displacements for NPT trajectories.

   :param coords: Fractional coordinates for all atoms.
   :param latt: Lattice descriptions.

   :return: Numpy array with shape [site, timestep, axis] describing displacements.
   """
   coords = np.concatenate(coords, axis=1)
   wrapped = np.einsum('ijk,jkl->ijk', coords, latt)
   wrapped_diff = np.diff(wrapped, axis=1)
   latt_para = np.einsum('ijj->ij', latt)

   unwrapped_disp = wrapped_diff - (np.floor(wrapped_diff / latt_para[1:] + 1 / 2) * latt_para[1:])

   return np.cumsum(unwrapped_disp, axis=1)


def get_disp_npt_matrix(coords, latt):
   """
   Calculate displacements for NPT trajectories.

   :param coords: Fractional coordinates for all atoms.
   :param latt: Lattice descriptions.

   :return: Numpy array with shape [site, timestep, axis] describing displacements.
   """
   coords = np.concatenate(coords, axis=1)
   latt_inv = np.linalg.inv(latt)
   wrapped = np.einsum('ijk,jkl->ijk', coords, latt)
   wrapped_diff = np.diff(wrapped, axis=1)

   unwrapped_disp = wrapped_diff - (np.einsum('ijk,jkl->ijk', np.floor(np.einsum('ijk,jkl->ijk', wrapped_diff, latt_inv[1:]) + (1 / 2)), latt[1:]))

   return np.cumsum(unwrapped_disp, axis=1)

In [11]:
coords = np.array([[[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1]],
                   [[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1],[0.1,0.1,0.1]],
                   [[0.2,0.2,0.2],[0.2,0.2,0.2],[0.2,0.2,0.2],[0.2,0.2,0.2]],
                   [[0.3,0.2,0.2],[0.3,0.2,0.2],[0.3,0.2,0.2],[0.3,0.2,0.2]],
                   [[0.4,0.2,0.2],[0.4,0.2,0.2],[0.4,0.2,0.2],[0.4,0.2,0.2]]
                   ])
coords = np.expand_dims(coords, 2)

latt = np.array([[[10,0,0],[0,10,0],[0,0,10]],
                 [[10,0,0],[0,10,0],[0,0,10]],
                 [[10,0,0],[0,10,0],[0,0,10]],
                 [[10,0,0],[0,10,0],[0,0,10]],
                 [[10,0,0],[0,10,0],[0,0,10]],
                ])

a = 9.758 
b = 10.499 
c = 12.205 
alpha = 108.58
beta  = 102.92
gamma = 82.52

a,b,c,alpha,beta,gamma = 12.046, 8.147, 7.548, 90, 120, 90

latt = np.array([_get_matrix([a,b,c,alpha,beta,gamma])])
latt = latt.repeat(5, axis=0)
print(latt)

"""
latt = np.array([[[2.4938607 , 2.45470387, 3.11596494],
                  [1.22846553, 8.72780397, 5.29343803],
                  [6.1614991 , 8.76431302, 7.2117923 ]],
                 [[2.4938607 , 2.45470387, 3.11596494],
                  [1.22846553, 8.72780397, 5.29343803],
                  [6.1614991 , 8.76431302, 7.2117923 ]],
                 [[2.4938607 , 2.45470387, 3.11596494],
                  [1.22846553, 8.72780397, 5.29343803],
                  [6.1614991 , 8.76431302, 7.2117923 ]],
                 [[2.4938607 , 2.45470387, 3.11596494],
                  [1.22846553, 8.72780397, 5.29343803],
                  [6.1614991 , 8.76431302, 7.2117923 ]],
                 [[2.4938607 , 2.45470387, 3.11596494],
                  [1.22846553, 8.72780397, 5.29343803],
                  [6.1614991 , 8.76431302, 7.2117923 ]]])
"""
print(coords.shape, latt.shape)

[[[ 1.04321420e+01  0.00000000e+00 -6.02300000e+00]
  [ 1.31013752e-15  8.14700000e+00  4.98859874e-16]
  [ 0.00000000e+00  0.00000000e+00  7.54800000e+00]]

 [[ 1.04321420e+01  0.00000000e+00 -6.02300000e+00]
  [ 1.31013752e-15  8.14700000e+00  4.98859874e-16]
  [ 0.00000000e+00  0.00000000e+00  7.54800000e+00]]

 [[ 1.04321420e+01  0.00000000e+00 -6.02300000e+00]
  [ 1.31013752e-15  8.14700000e+00  4.98859874e-16]
  [ 0.00000000e+00  0.00000000e+00  7.54800000e+00]]

 [[ 1.04321420e+01  0.00000000e+00 -6.02300000e+00]
  [ 1.31013752e-15  8.14700000e+00  4.98859874e-16]
  [ 0.00000000e+00  0.00000000e+00  7.54800000e+00]]

 [[ 1.04321420e+01  0.00000000e+00 -6.02300000e+00]
  [ 1.31013752e-15  8.14700000e+00  4.98859874e-16]
  [ 0.00000000e+00  0.00000000e+00  7.54800000e+00]]]
(5, 4, 1, 3) (5, 3, 3)


In [12]:
nvt_disp = get_disp(coords, latt)
npt_disp = get_disp_npt(coords, latt)
npt_matrix_disp = get_disp_npt_matrix(coords, latt)

np.testing.assert_array_almost_equal(npt_matrix_disp, nvt_disp)

In [14]:
import pandas as pd

lattices = pd.read_csv('lattice_params.csv', header=0)

for row in lattices.iterrows():
   row = row[1]
   lattice = np.array([_get_matrix([row['a'], row['b'], row['c'], row['alpha'], row['beta'], row['gamma']])])
   lattice = lattice.repeat(5, axis=0)

   ref_coords = np.random.rand(1,4,1,3)
   ref_coords = ref_coords.repeat(2, axis=0)
   rand_coords = np.random.rand(3,4,1,3)
   rand_coords = np.concatenate((ref_coords, rand_coords))

   nvt_disp = get_disp(rand_coords, lattice)
   npt_disp = get_disp_npt(rand_coords, lattice)
   npt_matrix_disp = get_disp_npt_matrix(rand_coords, lattice)
   np.testing.assert_array_almost_equal(npt_matrix_disp, nvt_disp)

AssertionError: 
Arrays are not almost equal to 6 decimals

Mismatched elements: 4 / 48 (8.33%)
Max absolute difference: 3.88141318
Max relative difference: 2.99783188
 x: array([[[ 0.      ,  0.      ,  0.      ],
        [-0.048164,  1.9706  , -2.686483],
        [ 1.351694, -0.698367, -1.085176],...
 y: array([[[ 0.      ,  0.      ,  0.      ],
        [-0.048164,  1.9706  , -2.686483],
        [ 1.351694, -0.698367, -1.085176],...

In [6]:
print(get_disp(coords, latt))

[[[ 0.          0.          0.        ]
  [ 0.41463035 -0.11303781  0.13964885]
  [ 0.8292607  -0.11303781  0.13964885]
  [ 1.24389104 -0.11303781  0.13964885]]

 [[ 0.          0.          0.        ]
  [ 0.41463035 -0.11303781  0.13964885]
  [ 0.8292607  -0.11303781  0.13964885]
  [ 1.24389104 -0.11303781  0.13964885]]

 [[ 0.          0.          0.        ]
  [ 0.41463035 -0.11303781  0.13964885]
  [ 0.8292607  -0.11303781  0.13964885]
  [ 1.24389104 -0.11303781  0.13964885]]

 [[ 0.          0.          0.        ]
  [ 0.41463035 -0.11303781  0.13964885]
  [ 0.8292607  -0.11303781  0.13964885]
  [ 1.24389104 -0.11303781  0.13964885]]]


In [7]:
print(get_disp_npt(coords, latt))

[[[ 0.          0.          0.        ]
  [ 0.41463035 -0.11303781  0.13964885]
  [ 0.8292607  -0.11303781  0.13964885]
  [ 1.24389104 -0.11303781  0.13964885]]

 [[ 0.          0.          0.        ]
  [ 0.41463035 -0.11303781  0.13964885]
  [ 0.8292607  -0.11303781  0.13964885]
  [ 1.24389104 -0.11303781  0.13964885]]

 [[ 0.          0.          0.        ]
  [ 0.41463035 -0.11303781  0.13964885]
  [ 0.8292607  -0.11303781  0.13964885]
  [ 1.24389104 -0.11303781  0.13964885]]

 [[ 0.          0.          0.        ]
  [ 0.41463035 -0.11303781  0.13964885]
  [ 0.8292607  -0.11303781  0.13964885]
  [ 1.24389104 -0.11303781  0.13964885]]]


In [8]:
print(get_disp_npt_matrix(coords, latt))

[[[ 0.          0.          0.        ]
  [ 0.41463035 -0.11303781  0.13964885]
  [ 0.8292607  -0.11303781  0.13964885]
  [ 1.24389104 -0.11303781  0.13964885]]

 [[ 0.          0.          0.        ]
  [ 0.41463035 -0.11303781  0.13964885]
  [ 0.8292607  -0.11303781  0.13964885]
  [ 1.24389104 -0.11303781  0.13964885]]

 [[ 0.          0.          0.        ]
  [ 0.41463035 -0.11303781  0.13964885]
  [ 0.8292607  -0.11303781  0.13964885]
  [ 1.24389104 -0.11303781  0.13964885]]

 [[ 0.          0.          0.        ]
  [ 0.41463035 -0.11303781  0.13964885]
  [ 0.8292607  -0.11303781  0.13964885]
  [ 1.24389104 -0.11303781  0.13964885]]]
